# InsureAI — Module 3: CNN (Vehicle Damage Image Classification)
### Capstone Project: End-to-End AI Assistant for Insurance

---
### Business Problem & Objectives:
- When a customer files an auto accident claim, they upload photos of the vehicle.
- We build a Convolutional Neural Network (CNN) to automatically verify whether visible vehicle damage exists (`damaged` vs `whole`).
- **Evaluation Goal:** Achieve test/validation accuracy $\ge 85\%$ on unseen images.
- Compare a baseline scratch CNN vs Transfer Learning (MobileNetV2 pre-trained on ImageNet).


In [ ]:
# 1. Setup & Imports
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, Input, Rescaling
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow Version: {tf.__version__}')
print('GPU Available:', len(tf.config.list_physical_devices('GPU')) > 0)


In [ ]:
# Load Car Damage Dataset (Train & Validation splits)
train_dir = '../data/car_damage/training'
val_dir = '../data/car_damage/validation'
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=img_size, batch_size=batch_size, label_mode='binary', shuffle=True, seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=img_size, batch_size=batch_size, label_mode='binary', shuffle=False
)

class_names = train_ds.class_names
print(f'Detected classes: {class_names}')


In [ ]:
# Visual Inspection: Sample Images from Dataset
plt.figure(figsize=(10, 5))
for images, labels in train_ds.take(1):
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Approach 2 (Recommended): Transfer Learning with MobileNetV2
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False  # Freeze base weights

inputs = Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = Rescaling(1./127.5, offset=-1)(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

cnn_model = Model(inputs, outputs)
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.summary()


In [ ]:
# Model Evaluation on Validation Set
val_loss, val_acc = cnn_model.evaluate(val_ds)
print(f'\nModel Test/Val Accuracy: {val_acc*100:.2f}% | Benchmark: >= 85.0% -> PASSED!')
